Вам предоставлена БД с логами (действиями) студентов на образовательном портале за весенний семестр (агрегация по каждой неделе) по отдельному электронному курсу - таблица user_logs (примечание. создана в предыдущих л.р.).
- сourseid — уникальный идентификатор курса, дисциплины;
- userid — уникальный идентификатор студента (не используется в обучении);
- num_week — номер недели в году;
- s_all — количество всех событий на текущий момент;
- s_all_avg — среднее количество всех событий в неделю;
- s_course_viewed — количество просмотров курса;
- s_course_viewed_avg — среднее количество просмотров курса в неделю;
- s_q_attempt_viewed — количество просмотров теста;
- s_q_attempt_viewed_avg — среднее количество просмотров теста в неделю;
- s_a_course_module_viewed — количество просмотров модуля в курсе;
- s_a_course_module_viewed_avg — среднее количество просмотров модуля в курсе в неделю;
- s_a_submission_status_viewed — количество отправленных заданий на проверку;
- s_a_submission_status_viewed_avg — среднее количество ответов;
- namer_level — оценка за дисциплину;
- depart — номер кафедры;
- name_osno — основа обучения (имеет два значения: бюджет или контракт);
- name_formopril — форма обучения;
- leveled — уровень образования (имеет два значения: бакалавриат, магистратура);
- num_sem — номер семестра;
- kurs — номер курса учебной группы.

Также в таблице  departments хранятся названия кафедр, таблица связана с логами по полю depart:
id - код кафедры;
name - сокращенное название кафедры.

In [ ]:
from dotenv import load_dotenv
%pip install seaborn matplotlib plotly pandas

# Визуализация данных с использованием Plotly

Plotly – это библиотека с открытым исходным кодом для Python и R, которая отлично подходит для создания красивых и интерактивных визуализаций.

Plotly — библиотека для визуализации данных, состоящая из нескольких частей:
- Front-End на JS
- Back-End на Python (за основу взята библиотека Seaborn)
- Back-End на R

Plotly позволяет строить интерактивные визуализации. 

Документация:

- [Plotly](https://plotly.com/)

- [Plotly.Express](https://plotly.com/python/plotly-express/)

- [Plotly.Dash](https://dash.plotly.com/) - построение дашбордов



In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy.engine import URL
import os
from dotenv import load_dotenv

In [ ]:
load_dotenv()
host = os.getenv("DB_HOST") or 'localhost'  # если переменная не установлена, используем localhost
port = os.getenv("DB_PORT")
db = os.getenv("DB_NAME")
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")

conn = URL.create(
    "postgresql",
    username=user,
    password=password,
    host=host,
    database=db,
    port=port
).render_as_string(hide_password=False)

df_user_logs = pd.read_sql_query("SELECT * FROM user_logs", con=conn, parse_dates=['date_vatt'])
df_departments = pd.read_sql_query("SELECT * FROM departments", con=conn)
df_user_logs.head(100)

In [ ]:
df_user_logs.info()

## Задание 1: Проведите предобработку набора данных

Преобразуйте столбцы, содержащие вещественные значения к типу float.


#### Поскольку использована загрузка из sql БД где данные уже предобработаны, данная предобработка не требуется.

## Примеры визуализаций
Построим столбчатую диаграмму количества студентов по каждому семестру. 
- Метод px.bar() - передаем данные для построения диаграммы
- Функция fig.update_layout() - настройки общего оформления графика (layout): заголовки осей, легенда, фон, отступы, шрифты и другие параметры визуализации.

In [ ]:
# Группируем по семестру и считаем количество уникальных студентов
df_grouped = df_user_logs.groupby("num_sem")["userid"].nunique().reset_index()
df_grouped.rename(columns={"userid": "num_students"}, inplace=True)

# Строим столбчатую диаграмму
fig = px.bar(
    df_grouped,
    x="num_sem",
    y="num_students",
    text="num_students",  # Подписи с количеством студентов
    color="num_students",
    title="Количество студентов, изучающих дисциплины в семестр",
    labels={"Num_Sem": "Семестр", "num_students": "Количество студентов"},
    color_continuous_scale="viridis"
)

fig.update_layout(
    xaxis_title="Номер семестра",
    yaxis_title="Количество студентов",
    template="plotly_white"
)

fig.show()

Построим точечную диаграмму количества студентов по каждому семестру. 

px.scatter() - передаем данные для построения диаграммы. 

In [ ]:
# Группируем по семестру и считаем количество уникальных студентов
df_grouped = df_user_logs.groupby("num_sem")["userid"].nunique().reset_index()
df_grouped.rename(columns={"userid": "num_students"}, inplace=True)

# Строим точечную диаграмму (scatter plot)
fig = px.scatter(
    df_grouped,
    x="num_sem",
    y="num_students",
    text="num_students",  # Подписи значений
    color="num_students",
    size="num_students",  # Размер точек зависит от количества студентов
    title="Количество студентов, изучающих дисциплины в семестр",
    labels={"Num_Sem": "Семестр", "num_students": "Количество студентов"},
    color_continuous_scale="viridis"
)

fig.update_layout(
    xaxis_title="Номер семестра",
    yaxis_title="Количество студентов",
    template="plotly_white"
)

# Добавляем подписи точек выше самих точек
fig.update_traces(
    textposition='top center',
    marker=dict(
        line=dict(width=1, color='DarkSlateGrey')
    )
)

fig.show()

Построим интерактивный график активности студентов по неделям
Используем plotly.express.area для построения графика изменения активности студентов (s_all) по неделям (num_week). Дополнительно, разделим данные по курсам (Kurs), чтобы увидеть, как активность различается среди студентов разных курсов.
Построить график px.area, где:
- Ось X — num_week (номер недели семестра)
- Ось Y — s_all (общее количество событий)
- Цветовая группировка — Kurs (номер курса студента)
- Добавить анимацию по семестрам (Num_Sem), чтобы увидеть динамику активности студентов разных курсов (свойство -  animation_frame).

In [ ]:
# Преобразуем колонки с числами, если они хранятся как строки
numeric_cols = ["s_all", "s_all_avg", "s_course_viewed", "s_course_viewed_avg",
                "s_q_attempt_viewed", "s_q_attempt_viewed_avg", "s_a_course_module_viewed",
                "s_a_course_module_viewed_avg", "s_a_submission_status_viewed", "s_a_submission_status_viewed_avg"]
for col in numeric_cols:
    df_user_logs[col] = pd.to_numeric(df_user_logs[col], errors="coerce")  # Преобразуем к float

# Строим график активности студентов по неделям
fig = px.area(
    df_user_logs,
    x="num_week",
    y="s_all",
    color="kurs",  # Разделение по курсам
    line_group="kurs",
    title="Динамика активности студентов в течение семестра",
    labels={"num_week": "Неделя", "s_all": "Количество событий", "kurs": "Курс"},
    animation_frame="num_sem"  # Анимация по семестрам
)

fig.show()

## Задание 2. Влияние формы обучения на активность
Решите гипотезу: студенты, обучающиеся на контракте, менее активны на портале по сравнению с бюджетниками.

Что визуализировать: 
Различие в среднем числе взаимодействий между группами ("Name_OsnO") по следующим показателями: "s_all_avg", "s_course_viewed_avg", "s_q_attempt_viewed_avg".
Необходимо использовать в  plotly.express Boxplot распределения активности для бюджетников и контрактников.

In [ ]:
import plotly.express as px

# 1 - бюджет, 2 - контракт
osno_mapping = {1: 'Бюджет', 2: 'Контракт'}
df_user_logs['osno_label'] = df_user_logs['name_osno'].map(osno_mapping)

# 2. "Плавим" таблицу для удобного отображения трех метрик на одном графике
metrics = ["s_all_avg", "s_course_viewed_avg", "s_q_attempt_viewed_avg"]
df_melted = df_user_logs.melt(
    id_vars=['osno_label'],
    value_vars=metrics,
    var_name='Metric',
    value_name='Value'
)

# 3. Строим Boxplot
fig = px.box(
    df_melted,
    x="osno_label",
    y="Value",
    color="osno_label",
    facet_col="Metric",  # Разделяем на три графика по вертикали
    points="outliers",  # Показывать только выбросы (можно поставить "all", если данных мало)
    title="Бюджет против Контракт",
    labels={"osno_label": "Форма обучения", "Value": "Среднее количество действий"},
    category_orders={"osno_label": ["Бюджет", "Контракт"]}
)

# Настройка внешнего вида
fig.update_layout(
    showlegend=False,
    template="plotly_white"
)

# Улучшаем читаемость названий подграфиков
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.show()

#### Ответ: Студенты, учащиеся по контракту действительно меньше взаимодействуют с порталом


# Задание 3. Динамика активности студентов по неделям
Гипотеза: Активность студентов увеличивается перед сессией и уменьшается после неё.

Постройте линейный график изменения активности (s_all) по неделям (num_week).
Сравнить активность по курсам (kurs). Все курсы должны отображаться на одной диаграмме. 

In [ ]:
import pandas as pd
import plotly.express as px

# Группируем по номеру недели и курсу, вычисляем среднее количество событий (s_all)
activity_dynamics = df_user_logs.groupby(['num_week', 'kurs'])['s_all'].mean().reset_index()  # убираем индексы
# неясно что за 6-7 курс, ну предположим это магистратура 1 и 2 курса
course_labels = {
    1: '1 курс (Бак/Спец)',
    2: '2 курс (Бак/Спец)',
    3: '3 курс (Бак/Спец)',
    4: '4 курс (Бак/Спец)',
    5: '5 курс (Спец)',
    6: '1 курс (Маг)',
    7: '2 курс (Маг)'
}
# Если курса нет в словаре, он оставит старое значение (на всякий случай потому что не понятно что там вообще за курсы)
activity_dynamics['kurs_name'] = activity_dynamics['kurs'].map(course_labels).fillna(activity_dynamics['kurs'].astype(str) + ' курс')
# Построение графика
fig = px.line(
    activity_dynamics,
    x='num_week',
    y='s_all',
    color='kurs_name',
    title='Динамика средней активности студентов по неделям',
    labels={
        'num_week': 'Номер недели',
        's_all': 'Средняя активность студентов',
        'kurs_name': 'Курс обучения'
    },
    markers=True
)

fig.update_layout(
    xaxis=dict(tickmode='linear', dtick=1),  # Шаг оси X равен 1 чтобы видеть каждую неделю
    hovermode='x unified'  # Отображение данных при наведении
)

fig.show()

Ваш ответ: Да, это определенно так.

## Задание 4: Проанализировать динамику активности студентов с разным уровнем успеваемости (NameR_Level) в течение всего семестра.
Решите гипотезу: успеваемость студентов зависит от динамики их активности на образовательном портале в течение всего семестра. 

Построить линейный график (px.line), где:
X-ось — num_week (номер недели семестра)
Y-ось — s_all (общее количество событий)
Цветовая группировка — NameR_Level (успеваемость: отличники, хорошисты, троечники, двоечники)

In [ ]:

# Считаем среднюю активность, группируем по неделям и уровню успеваемости
activity_by_grade = df_user_logs.groupby(['num_week', 'namer_level'])['s_all'].mean().reset_index()

# Метки
grade_labels = {
    5: 'Отличники',
    4: 'Хорошисты',
    3: 'Троечники',
    2: 'Двоечники'
}
activity_by_grade['grade_name'] = activity_by_grade['namer_level'].map(grade_labels).fillna(activity_by_grade['namer_level'].astype(str))

# Построение графика
fig = px.line(
    activity_by_grade,
    x='num_week',
    y='s_all',
    color='grade_name',
    markers=True,
    title='Активность студентов по уровням успеваемости',
    labels={
        'num_week': 'Номер недели',
        's_all': 'Средняя активность',
        'grade_name': 'Успеваемость'
    }
)
fig.update_layout(
    xaxis=dict(dtick=1),  # Показываем каждую неделю
    hovermode='x unified'  # При наведении видим данные всех групп
)

fig.show()

Ваш ответ: Да, это так, в среднем в течении семестра чем выше активность - тем лучше успеваемость

# Задание 5. Различие в активности бакалавров и магистров
Гипотеза: Магистранты взаимодействуют с курсами иначе, чем бакалавры.

Построить воронку (метод go.Funnel()) активности студентов (s_course_viewed, s_q_attempt_viewed, s_a_submission_status_viewed) в зависимости от уровня образования (leveled).

In [ ]:
# типы обучения в словарь чтобы потом по циклу пройти
level_map = {
    1: "бакалавриат",
    2: "магистратура",
    3: "специалитет",
    4: "аспирантура"
}

df_user_logs["leveled"] = df_user_logs["leveled"].replace(level_map)

# Группируем данные
df_level = df_user_logs.groupby("leveled")[[
    "s_course_viewed",
    "s_q_attempt_viewed",
    "s_a_submission_status_viewed"
]].mean().reset_index()

# Подписи оси Y
stages = ["Просмотры курса", "Просмотры тестов", "Отправка заданий"]

# Строим воронку
fig = go.Figure()

for level_name in level_map.values():

    fig.add_trace(go.Funnel(
        name=level_name.capitalize(),
        y=stages,
        x=df_level[df_level["leveled"] == level_name].iloc[:, 1:].values.flatten()
    ))

fig.update_layout(title="Различие в активности бакалавров и магистров")

fig.show()

Ответ: Я не заметил разницы, магистранты также как бакалавриат взаимодействуют, отличия есть у аспирантов только.ё

## Задание 6. Различие в активности отличников, хорошистов, троечников и двоечников 
Гипотеза: Студенты с разным уровнем успеваемостью взаимодействуют с курсами иначе.

Построить воронку активности студентов (s_course_viewed, s_q_attempt_viewed, s_a_submission_status_viewed) в зависимости от уровня успеваемости (NameR_Level).

In [ ]:
import plotly.graph_objects as go

# Подготовка имен для категорий успеваемости
grade_map = {5: "Отличники", 4: "Хорошисты", 3: "Троечники", 2: "Двоечники"}
df_user_logs["grade_label"] = df_user_logs["namer_level"].map(grade_map)

# Группируем данные по уровню успеваемости и считаем среднее для каждого типа активности
df_grades = df_user_logs.groupby("grade_label")[["s_course_viewed", "s_q_attempt_viewed", "s_a_submission_status_viewed"]].mean().reset_index()

# Названия этапов воронки
stages = ["Просмотры курса", "Просмотры тестов", "Отправка заданий"]

# 4. Строим воронку
fig = go.Figure()

for grade in grade_map.values():
    fig.add_trace(go.Funnel(
        name=grade,
        y=stages,
        x=df_grades[df_grades["grade_label"] == grade].iloc[:, 1:4].values.flatten(),
        textinfo="value+percent initial"
    ))

fig.update_layout(
    title="Воронка активности студентов в зависимости от успеваемости",
    yaxis_title="Этапы активности",
    legend_title="Уровень оценки"
)

fig.show()

Ответ: Каких-то сильных различий замечено не было, разве что относительно просмотров курсов двоечники мало смотрят тесты.

## Задание 7:  Распределение студентов по кафедрам 
Определим распределение студентов по кафедрам, однаковое ли количество студентов?

Постройте горизонтальную столбчатую диаграмму (bar chart) для сравнения количества студентов на разных кафедрах.
Добавим интерактивность, цветовую палитру и подписи. На диаграмме должны быть выведены названия кафедр (возьмите данную информацию из Task 4.)

In [ ]:
import plotly.express as px

# Объединяем таблицы (логи и кафедры)
# по сути left join на DataFrame
df_merged = df_user_logs.merge(df_departments, left_on='depart', right_on='id', how='left')


# Группируем по названию кафедры и считаем уникальных userid
dept_distribution = df_merged.groupby('name')['userid'].nunique().reset_index()
dept_distribution.columns = ['Кафедра', 'Количество студентов']

# Сортируем по возрастанию
dept_distribution = dept_distribution.sort_values(by='Количество студентов', ascending=True)

# Построение горизонтальной столбчатой диаграммы
fig = px.bar(
    dept_distribution,
    x='Количество студентов',
    y='Кафедра',
    orientation='h', # Горизонтальная ориентация
    text='Количество студентов', # Подписи на столбцах
    color='Количество студентов', # Цветовая палитра зависит от величины
    color_continuous_scale='Viridis',
    title='Распределение уникальных студентов по кафедрам',
    labels={'Количество студентов': 'Число студентов', 'Кафедра': 'Название кафедры'}
)

# Настройка оформления
fig.update_traces(textposition='outside')
fig.update_layout(
    showlegend=False,
    height=800, # Увеличиваем высоту, кафедр сильно много
    xaxis_title="Количество уникальных студентов",
    yaxis_title=None
)

fig.show()

## Задание 8: Зависимость среднего балла от количества студентов на кафедре
Гипотеза: количество студентов, закрепленных на отдельной кафедре влияет на средний балл успеваемости по кафедрам.

Построить диаграмму рассевания по количеству студентов, среднему баллу, наванию кафедры. Параметр  size='num_students' - количество студентов. 

In [ ]:
import plotly.express as px

# 1. Объединяем данные логов с названиями кафедр
df_merged = df_user_logs.merge(df_departments, left_on='depart', right_on='id', how='left')

# 2. Подготовка данных по кафедрам
# Сначала получаем уникальные записи (студент, его оценка и его кафедра)
unique_students_grades = df_merged.drop_duplicates(subset=['userid', 'courseid'])

# Группируем по названию кафедры
dept_stats = unique_students_grades.groupby('name').agg(
    num_students=('userid', 'nunique'),
    avg_grade=('namer_level', 'mean')
).reset_index()

# 3. Построение диаграммы рассеяния (Scatter Plot)
fig = px.scatter(
    dept_stats,
    x='num_students',
    y='avg_grade',
    size='num_students',      # Размер пузырька зависит от кол-ва студентов
    color='avg_grade',        # Цвет зависит от среднего балла
    hover_name='name',        # При наведении показываем название кафедры
    text='name',              # Добавляем подписи названий прямо на график
    title='Связь среднего балла и количества студентов по кафедрам',
    labels={
        'num_students': 'Количество студентов на кафедре',
        'avg_grade': 'Средний балл успеваемости',
        'name': 'Кафедра'
    },
    color_continuous_scale='RdYlGn' # Цветовая схема: от красного к зеленому
)

# Улучшаем читаемость подписей
fig.update_traces(textposition='top center')
fig.update_layout(
    height=600,
    xaxis_title="Число студентов",
    yaxis_title="Средний балл (namer_level)"
)

fig.show()

Ваш ответ: На мой взгляд никакой корреляции не прослеживается
